# The sim3d engine, end to end

Every stage of the reservoir-to-seismic chain in one notebook, on the
40-degree dipping trap: geology, wells, three years of flow, rock physics,
the 1D-convolution cube, 4D time shifts, survey noise, the acquisition and
the migration.

Nothing here is notebook-specific. `sim3d.ui.components` imports only numpy
and plotly - no Streamlit - so these are the same figures the app draws,
from the same `Pipeline` the CLI uses. The app's own docstring puts it
plainly: *"Streamlit is only a frontend. No scientific logic lives here."*

**Why a notebook suits this.** `Pipeline` caches each stage on the instance,
so `p.flow()` runs once and every later cell reuses it. Running these stages
as separate scripts recomputes geology, flow and rock physics every time -
minutes apiece, for a result that never changed.

## Setup

In [ ]:
import numpy as np
import plotly.io as pio

from sim3d.core.config import ExperimentConfig
from sim3d.core.units import pa_to_psi, si_to_stb_per_day
from sim3d.experiments.pipeline import Pipeline
from sim3d.fourd.metrics import nrms
from sim3d.fourd.scenarios import build_earth_models, build_states_from_flow
from sim3d.processing.preview import time_from_depth
from sim3d.processing.sim2seis import sim2seis_volume
from sim3d.ui import components as ui
from sim3d.ui import theme

pio.renderers.default = "notebook"   # or "plotly_mimetype+notebook_connected" in Lab
CONFIG = "../examples/configs/dipping_wedge_4d.yaml"

## 1. The configuration

Everything that affects a result lives in the YAML: a typo in a key is an
error rather than a silently-defaulted setting.

In [ ]:
cfg = ExperimentConfig.load(CONFIG)
p = Pipeline(cfg)
print(cfg.project.description.strip())
print()
print(f"dip            {cfg.geology.parameters['dip']} deg")
print(f"sand           {cfg.geology.parameters['gross']} m at "
      f"{cfg.geology.parameters['z_reservoir']} m")
print(f"OWC            {cfg.reservoir.baseline.owc} m, "
      f"{cfg.reservoir.baseline.transition} m transition")
print(f"wells          {[(w['name'], w['role'], w['x']) for w in cfg.wells.wells]}")
print(f"duration       {cfg.simulation.duration_days} days")
print(f"source         {cfg.source.type} {cfg.source.corners or cfg.source.frequency}")
print(f"Fmax           {p.fmax:.1f} Hz -> seismic dt {p.seismic_sample_interval()*1000:.0f} ms")

## 2. Geology

Three units - shale, sand, shale - tilted 40 degrees about the model centre.
The whole package dips together: tilting only the reservoir would drive it
up through a flat seal, which is a crossing horizon rather than a structure.

**The cell size is set by the dip, not by the seismic.** A bed of thickness
`g` dipping at `theta` drops `dx*tan(theta)` per lateral cell, so adjacent
columns of sand overlap by only `g - dx*tan(theta)`. At 40 degrees with
`dx = 20` m and a 20 m bed that is 3 m, and a third of the column pairs then
share no cell at all - the reservoir becomes a disconnected staircase, the
wells lose pressure communication, and *the material balance still closes*.

In [ ]:
g = p.geology()
grid = g.grid
cursor = (500.0, 700.0, 2000.0)
print(f"grid {grid.shape} = {grid.n_cells:,} cells, spacing {grid.spacing}")
print(f"reservoir cells {int(g.reservoir_mask.sum()):,}")
ui.slice_figure(g.porosity, grid, cursor, title="porosity",
                kind="sequential", unit="fraction", wells=p.wells())

In [ ]:
# The connectivity check that the staircase failure motivated.
from sim3d.validation.qc import check_layer_connectivity
for c in check_layer_connectivity(g).checks:
    print(c.status.value, c.message)

## 3. Wells and fluid contacts

P1 sits updip in the oil leg, I1 downdip in the water leg. Completions are
chosen by geological unit, so the depths follow the structure rather than
being typed in - which is what makes them stay correct on a 40-degree dip.

In [ ]:
from sim3d.wells.completion import layer_intersections
for well in p.wells():
    print(well.name)
    for u in layer_intersections(well, g):
        if u.is_reservoir:
            print(f"   {u.name:12s} {u.top:7.0f}-{u.base:7.0f} m  "
                  f"{u.net_thickness:5.1f} m net  {u.permeability_md:7.0f} mD")

In [ ]:
state = p.baseline_state()
print(" | ".join(state.provenance))
iy = int(np.argmin(np.abs(grid.axis(1) - 700.0)))
res = g.reservoir_mask
sw = np.where(res, state.sw, np.nan)
profile = {"Sw in the sand": np.array(
    [np.nanmean(sw[i, iy]) if res[i, iy].any() else np.nan
     for i in range(grid.nx)])}
ui.series_figure(grid.axis(0), profile, xlabel="x (m)   P1 -> I1, downdip ->",
                 ylabel="water saturation", height=320)

## 4. Three years of flow

Water into the water leg downdip, oil out updip. The contact is pushed
updip; the producer sees no water in three years.

In [ ]:
flow = p.flow()
print(f"{flow.n_timesteps} timesteps, material balance {flow.material_balance_error:.1e}")
for name, h in flow.wells.items():
    print(" ", h.summary())

In [ ]:
rates, days = {}, None
for name, h in flow.wells.items():
    a = h.arrays()
    if a["days"].size:
        days = a["days"]
        rates[f"{name} oil"] = si_to_stb_per_day(np.abs(a["oil_rate"]))
        rates[f"{name} water"] = si_to_stb_per_day(np.abs(a["water_rate"]))
ui.series_figure(days, rates, xlabel="day", ylabel="rate (STB/day)", height=380)

In [ ]:
states = p.reservoir()
dsw = np.where(res, states.combined.sw - states.baseline.sw, np.nan)
ui.slice_figure(dsw, grid, cursor, title="change in water saturation, 3 years",
                kind="diverging", unit="fraction", wells=p.wells())

## 5. Rock physics

**One global dry frame is a bias, not a simplification.** With a single
soft-sand frame the shale comes out *slower* than the reservoir sand, so the
impedance step at the top of the sand has the wrong sign and almost no size.
A stiff frame for the shale restores it. That is what `rock_physics.facies`
is for; fluid properties stay global, because one connected reservoir has
one fluid.

In [ ]:
earth = p.rockphysics()
rp = earth.rock_physics["baseline"]
print(f"Vp  sand {rp.vp[res].mean():7.0f}   shale {rp.vp[~res].mean():7.0f} m/s")
print(f"AI  sand {(rp.vp*rp.rho)[res].mean():9.0f}   shale {(rp.vp*rp.rho)[~res].mean():9.0f}")
ui.slice_figure(rp.ai, grid, cursor, title="acoustic impedance - baseline",
                kind="sequential", unit="", wells=p.wells())

## 6. The 1D-convolution cube (sim2seis)

The earth model turned into a synthetic volume, column by column, in angle
stacks. Seconds, not hours, because nothing is propagated and nothing is
migrated - and therefore **never an image**: every trace is built
independently of its neighbours, so structure steeper than a few degrees is
mispositioned. On a 40-degree dip that matters, and it is exactly what the
migration in section 9 exists to fix.

In [ ]:
vols = p.sim2seis()
base = vols["baseline"]
print(base.describe())
print()
for scenario in ("pressure_only", "saturation_only", "combined"):
    row = [f"{nrms(vols[scenario].time_cube(s), base.time_cube(s)):6.2f}"
           for s in base.names]
    print(f"NRMS {scenario:16s} " + "  ".join(row) + "   (near mid far, %)")

The saturation response *brightens* with angle and the pressure response
*dims*. That opposition is the discriminator the angle stacks exist for: a
far-stack difference that brightens is fluid, one that dims is pressure.

## 7. A section between the wells, through time

The wells share y = 700 m, so the P1 -> I1 line is a constant-y inline and
the section is a straight slice rather than an interpolated traverse.

One time axis for every survey, set by the slowest: each has its own
velocity and so its own deepest two-way time, and letting each end at its
own would put the monitors on axes the baseline cannot be subtracted from.

In [ ]:
DAYS = [0.0, 365.0, 730.0, 1095.0]
dt = p.seismic_sample_interval()
wavelet = p.wavelet(np.arange(int(2.0 / dt)) * dt)
baseline_state = p.baseline_state()

earths = {d: build_earth_models(build_states_from_flow(baseline_state, flow, d),
                                g, p.rock_physics_config(),
                                facies_configs=cfg.rock_physics.facies)
          for d in DAYS}
t_max = max(float(time_from_depth(e.rock_physics['combined'].vp, grid.dz).max())
            for e in earths.values())

sections = {}
for d in DAYS:
    r = earths[d].rock_physics['combined']
    vol = sim2seis_volume(grid, r.vp, r.vs, r.rho, wavelet, dt,
                          stacks=cfg.sim2seis.stacks, t_max=t_max,
                          map_to_depth=False)
    sections[d] = vol.time_cube('near')[:, iy, :]
    times = vol.times
print(f"{len(DAYS)} sections of {sections[DAYS[0]].shape} on a shared {t_max:.2f} s axis")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def section_panel(title, data, days, zmax):
    fig = make_subplots(rows=1, cols=len(days), shared_yaxes=True,
                        subplot_titles=[f'day {d:.0f}' for d in days],
                        horizontal_spacing=0.02)
    for i, d in enumerate(days):
        fig.add_trace(go.Heatmap(z=data[d].T, x=grid.axis(0), y=times, zmid=0.0,
                                 zmin=-zmax, zmax=zmax, colorscale=theme.DIVERGING,
                                 showscale=(i == len(days) - 1)), row=1, col=i + 1)
        for wx in (200.0, 800.0):
            fig.add_vline(x=wx, line=dict(color=theme.INK_SECONDARY, width=1,
                                          dash='dot'), row=1, col=i + 1)
        fig.update_xaxes(title_text='x (m)   P1 -> I1, downdip ->', row=1, col=i + 1)
    fig.update_yaxes(autorange='reversed', title_text='two-way time (s)', row=1, col=1)
    fig.update_layout(**theme.plotly_layout(height=560, title=title,
                                            margin=dict(l=70, r=20, t=70, b=60)))
    return fig

peak = float(np.abs(sections[0.0]).max())
section_panel('synthetic near stack along the well line', sections, DAYS, peak)

In [ ]:
diffs = {d: sections[d] - sections[0.0] for d in DAYS[1:]}
section_panel('4D difference from day 0', diffs, DAYS[1:],
              float(np.abs(diffs[DAYS[-1]]).max()))

**The peak grows, then saturates; the anomaly's edges say more than its
peak.** In the first year the anomaly is still filling in. After that,
wherever the front has passed the oil-to-water substitution is *complete*,
so the amplitude change there is maxed out and only the swept area grows.
Its downdip edge stays where the contact started; its updip edge advances
in step with the contact the flow simulation puts there.


In [ ]:
for d in DAYS[1:]:
    moved = int((np.abs(diffs[d]).max(axis=1) > 0.10 * peak).sum())
    contact = next((grid.axis(0)[i] for i in range(grid.nx)
                    if res[i, iy].any()
                    and np.mean(earths[d].states.combined.sw[i, iy][res[i, iy]]) > 0.6),
                   float('nan'))
    print(f"day {d:>6.0f}:  {moved:3d} traces moved   flow contact at x = {contact:.0f} m")

## 8. Time shifts and survey noise

A velocity change moves amplitudes where the rock changed *and* traveltimes
for everything beneath it. Differencing without accounting for the second
turns a shift into a derivative-shaped anomaly at the wrong depth.

The true shift is exact here, from the two-way-time cubes the volumes
already build; the estimate is what a windowed correlation recovers. Both
are kept, because their difference is the measurement error.

In [ ]:
shifts = p.time_shifts()
for note in p.result.notes:
    if 'time shift' in note:
        print(note)

In [ ]:
from sim3d.fourd.noise import NoiseModel
for level in (0.05, 0.10):
    for r in (0.0, 0.7):
        m = NoiseModel(level=level, repeatability=r)
        print(f"{m.describe()}")

The floor is `100 * level * sqrt(2 * (1 - r))` percent, and every 4D number
in the experiment has to beat it to mean anything. A noise-free synthetic
pair has an NRMS of zero, which is not a good result but a meaningless one.

## 9. Acquisition and the migration

The expensive path: propagate, acquire, migrate. Check the cost first - the
planner refuses anything past the budget and says what to change.

In [ ]:
acq = p.acquisition()
print(f"{acq.n_sources} sources, {acq.n_receivers} receivers, {acq.n_traces:,} traces")
for c in p.geometry_qc().checks:
    print(" ", c.status.value, c.message)
ui.aerial_figure(acquisition=acq, wells=p.wells(), domains=p.domains,
                 pml_nodes=cfg.solver.pml_nodes)

In [ ]:
# The migrated variant lowers the bandwidth to what the grid can carry and
# models two surveys rather than four. Run `sim3d benchmark` once first and
# the plan below prints a predicted runtime instead of leaving it blank.
rtm_cfg = ExperimentConfig.load('../examples/configs/dipping_wedge_rtm.yaml')
rtm = Pipeline(rtm_cfg)
print(f"Fmax {rtm.fmax:.1f} Hz")
print(rtm.plan().describe())

Dispersion is governed by the **coarsest** axis, not the axis the model
varies along: the wavefield propagates in y whether or not the geology does.
A 10 x 20 x 10 m grid sampled the 52.8 m minimum wavelength at 2.6 cells
where an 8th-order scheme needs 5.25, and QC failed it. Hence isotropic 10 m.

In [ ]:
# Hours, not minutes. Uncomment to run.
# rtm.simulate(progress=lambda n, d, c: print(f'fwd {n} {d}/{c}'))
# images = rtm.migrate(progress=lambda n, d, c: print(f'mig {n} {d}/{c}'))
# b, m = images['baseline'].image, images['combined'].image
# print(f'RTM 4D NRMS {nrms(m, b):.2f} %')
# ui.slice_figure(m - b, rtm.domains.propagation, cursor,
#                 title='RTM 4D difference', kind='diverging', unit='')

## What this notebook is not

Two things in the app do not carry over, both interaction rather than
science: **click-to-place wells and drawn channels**, which depend on
Streamlit's plot-selection events, and **buttons and progress bars**. Set
coordinates in code instead - `cfg.wells.wells.append({...})`,
`cfg.geology.bodies = [{...}]` - and the results are identical.

Everything numerical is config plus pipeline calls, and reaches the same
code the CLI and the app both run.